# Programação dinâmica: memoization e subestrutura ótima — Tutorial

**Algoritmos e Estruturas de Dados 2 — DCOMP/UFS**
Prof. Dr. André Yoshiaki Kashiwabara

Este tutorial acompanha a aula. O código dos algoritmos é escrito em C e compilado com `gcc`
a partir do próprio notebook: cada célula `%%writefile` grava um programa em `src/`, e a célula
seguinte compila e executa. As medições, os gráficos e alguns exercícios ficam em Python.

## Objetivos

Ao final deste tutorial você será capaz de:

- medir o custo exponencial da **recursão direta** de Fibonacci e explicá-lo pelos **subproblemas sobrepostos**;
- transformar uma recursão em **memoization** (top-down) e em **bottom-up**, contando as chamadas em cada caso;
- verificar, em um grafo pequeno, que caminho mínimo tem **subestrutura ótima** e caminho simples mais longo não tem;
- implementar a **seleção de intervalos ponderados** completa: vetor $p$ por busca binária, tabela $M$ e reconstrução da solução.

In [ ]:
# Preparação do ambiente: pasta de trabalho, compilador e funções auxiliares.
import os, subprocess

os.makedirs('src', exist_ok=True)
print(subprocess.run(['gcc', '--version'], capture_output=True, text=True).stdout.splitlines()[0])

_compilados = {}   # fonte -> data de modificação da última compilação bem-sucedida

def compilar(fonte):
    """Compila src/<fonte> com gcc -O2 (só se o arquivo mudou); devolve o executável ou None."""
    exe = os.path.join('src', os.path.splitext(fonte)[0])
    mtime = os.path.getmtime(os.path.join('src', fonte))
    if _compilados.get(fonte) == mtime and os.path.exists(exe):
        return exe
    comp = subprocess.run(['gcc', '-std=c99', '-Wall', '-O2', '-o', exe,
                           os.path.join('src', fonte), '-lm'],
                          capture_output=True, text=True)
    if comp.returncode != 0:
        print('ERRO DE COMPILAÇÃO:\n' + comp.stderr)
        return None
    if comp.stderr:
        print(comp.stderr)
    _compilados[fonte] = mtime
    return exe

def rodar(fonte, args=(), entrada=None, mostrar=True):
    """Compila src/<fonte>, executa com os argumentos e a entrada dados e devolve a saída."""
    exe = compilar(fonte)
    if exe is None:
        return ''
    r = subprocess.run([exe, *map(str, args)], input=entrada, capture_output=True, text=True)
    if mostrar:
        print(r.stdout, end='')
    if r.returncode != 0:
        print(f'(programa terminou com código {r.returncode})', r.stderr)
    return r.stdout

def campo(saida, nome):
    """Extrai o valor de 'nome=valor' da saída dos programas (int se for inteiro, senão float)."""
    for tok in saida.split():
        if tok.startswith(nome + '='):
            valor = tok.split('=')[1]
            return int(valor) if valor.lstrip('-').isdigit() else float(valor)
    raise ValueError(f'{nome} não encontrado em: {saida!r}')

def linha(saida, rotulo):
    """Devolve os inteiros da linha que começa com 'rotulo:' (ex.: 'p:', 'M:', 'solucao:')."""
    for lin in saida.splitlines():
        if lin.startswith(rotulo + ':'):
            return [int(x) for x in lin.split(':', 1)[1].split()]
    raise ValueError(f'linha {rotulo} não encontrada em: {saida!r}')

# Instância condutora dos slides: n, depois "s f v" de cada intervalo (já ordenados por término).
INSTANCIA = '6\n0 4 1\n1 6 6\n5 7 4\n2 9 7\n8 10 2\n7 11 1\n'

## 1. Subproblemas que se repetem: Fibonacci pela definição

$F_0 = 0$, $F_1 = 1$ e $F_n = F_{n-1} + F_{n-2}$. A recursão direta chama `fib(n-1)` e
`fib(n-2)`, e cada uma delas volta a chamar os mesmos valores menores. O número de chamadas
satisfaz $C(0)=C(1)=1$ e $C(n) = C(n-1) + C(n-2) + 1$, cuja solução é $C(n) = 2F_{n+1} - 1$:
cresce como $\phi^n$, com $\phi \approx 1{,}618$. Só existem $n+1$ subproblemas distintos.

O programa abaixo conta as chamadas e mede o tempo. Cada incremento de $n$ multiplica o tempo
por cerca de $\phi$; somar 5 a $n$ multiplica por $\phi^5 \approx 11$.

**Fonte:** Dasgupta et al. (2008), §0.2; Cormen et al. (2009), §15.3 (subproblemas sobrepostos).

In [ ]:
%%writefile src/fib_ingenuo.c
/* FIBONACCI PELA DEFINICAO: recursao direta, sem tabela.
   Uso: ./fib_ingenuo n      (imprime F(n), o numero de chamadas e o tempo)
   Dasgupta et al. (2008), sec. 0.2; Levitin (2012), cap. 8. */
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

static long long chamadas = 0;

long long fib( int n) {
   ++chamadas;
   if (n <= 1) return n;
   return fib( n-1) + fib( n-2);
}

int main( int argc, char *argv[]) {
   if (argc < 2) { fprintf( stderr, "uso: %s n\n", argv[0]); return 1; }
   int n = atoi( argv[1]);
   clock_t t0 = clock();
   long long f = fib( n);
   double seg = (double) (clock() - t0) / CLOCKS_PER_SEC;
   printf( "n=%d fib=%lld chamadas=%lld tempo=%.4f\n", n, f, chamadas, seg);
   return 0;
}

In [ ]:
# Exemplo: chamadas e tempo da recursão direta; confere C(n) = 2 F(n+1) - 1.
F = [0, 1]
while len(F) < 100:
    F.append(F[-1] + F[-2])

chamadas_ing, tempos_ing = {}, {}
for n in [5, 10, 15, 20, 25, 30, 35]:
    s = rodar('fib_ingenuo.c', args=(n,))
    assert campo(s, 'fib') == F[n] and campo(s, 'chamadas') == 2 * F[n + 1] - 1
    chamadas_ing[n], tempos_ing[n] = int(campo(s, 'chamadas')), campo(s, 'tempo')
print('C(n) = 2 F(n+1) - 1 conferido.')

In [ ]:
# Visualização: chamadas da recursão direta (escala log) contra phi^n.
import math
import matplotlib.pyplot as plt

phi = (1 + math.sqrt(5)) / 2
ns = sorted(chamadas_ing)
plt.figure(figsize=(6, 4))
plt.semilogy(ns, [chamadas_ing[n] for n in ns], 'o-', label='chamadas de fib(n)')
plt.semilogy(ns, [phi ** n for n in ns], '--', label=r'$\phi^n$')
plt.semilogy(ns, [n + 1 for n in ns], 's-', label='subproblemas distintos: n + 1')
plt.xlabel('n')
plt.ylabel('quantidade (escala log)')
plt.title('Recursão direta: exponencial; subproblemas distintos: linear')
plt.legend()
plt.grid(True, which='both', alpha=0.3)
plt.show()

In [ ]:
# Exercício 1: contando chamadas com programação dinâmica
# TODO: implemente chamadas(n), o número de chamadas que fib(n) faz, pela recorrência
#     C(0) = C(1) = 1,   C(n) = C(n-1) + C(n-2) + 1.
# Não use recursão direta (ela mesma seria exponencial!): preencha os valores de baixo para
# cima, de C(0) até C(n), guardando só o que for necessário.

def chamadas(n):
    # TODO: implemente aqui
    pass

# Teste automático
assert chamadas(0) == 1 and chamadas(1) == 1, 'Casos base: uma chamada'
assert chamadas(5) == 15, 'fib(5) faz 15 chamadas (árvore dos slides)'
assert chamadas(30) == 2692537, 'Verifique sua implementação'
assert chamadas(90) == 2 * F[91] - 1, 'Confira C(n) = 2 F(n+1) - 1'
print('Exercício 1 ok! fib(90) pela definição faria', f'{chamadas(90):,}', 'chamadas.')

## 2. Memoization e bottom-up

A **memoization** mantém a recursão e acrescenta uma tabela `memo[]`, iniciada com $-1$
("ainda não calculado"): antes de calcular, consulta; depois de calcular, anota. A versão
**bottom-up** preenche a tabela com um laço, na ordem $F_0, F_1, \dots, F_n$, em que cada
entrada só depende de entradas já prontas.

Com memoization, `fibMemo(n)` faz exatamente $2n - 1$ chamadas para $n \ge 1$: uma por
subproblema que é calculado, mais uma consulta para cada $F_k$ já anotado.
O programa abaixo vai até $n = 92$, o maior Fibonacci que cabe em `long long`.

**Fonte:** Cormen et al. (2009), §15.1 ("top-down with memoization" e "bottom-up method") e §15.3.

In [ ]:
%%writefile src/fib_pd.c
/* FIBONACCI COM PROGRAMACAO DINAMICA: memoization (top-down) e bottom-up.
   Uso: ./fib_pd n      (0 <= n <= 92; imprime F(n) pelas duas versoes)
   Cormen et al. (2009), secs. 15.1 e 15.3. */
#include <stdio.h>
#include <stdlib.h>

#define MAX 93                 /* F(92) e o maior Fibonacci que cabe em long long */

static long long memo[MAX];    /* memo[k] == -1: F(k) ainda nao calculado */
static long long chamadas = 0;

long long fibMemo( int n) {
   ++chamadas;
   if (memo[n] >= 0) return memo[n];          /* ja calculado: consulta */
   long long r;
   if (n <= 1) r = n;
   else r = fibMemo( n-1) + fibMemo( n-2);
   memo[n] = r;                               /* anota antes de devolver */
   return r;
}

long long fibBU( int n) {
   long long F[MAX];
   F[0] = 0;
   if (n >= 1) F[1] = 1;
   for (int k = 2; k <= n; ++k)               /* ordem: F[k-1] e F[k-2] ja prontos */
      F[k] = F[k-1] + F[k-2];
   return F[n];
}

int main( int argc, char *argv[]) {
   if (argc < 2) { fprintf( stderr, "uso: %s n\n", argv[0]); return 1; }
   int n = atoi( argv[1]);
   if (n < 0 || n >= MAX) { fprintf( stderr, "n fora de [0, %d]\n", MAX-1); return 1; }
   for (int k = 0; k < MAX; ++k) memo[k] = -1;
   long long a = fibMemo( n), b = fibBU( n);
   printf( "n=%d memo=%lld bottomup=%lld chamadas=%lld\n", n, a, b, chamadas);
   return 0;
}

In [ ]:
# Exemplo: as duas versões concordam e a memoizada faz 2n - 1 chamadas.
for n in [1, 2, 5, 30, 50, 92]:
    s = rodar('fib_pd.c', args=(n,))
    assert campo(s, 'memo') == campo(s, 'bottomup') == F[n]
    assert campo(s, 'chamadas') == 2 * n - 1
print(f'fib(92) = {F[92]:,} com 183 chamadas; pela definição seriam {2 * F[93] - 1:,}.')

### Exercício 2: de quantas maneiras subir uma escada?

Subindo 1, 2 ou 3 degraus por passo, o número $E(n)$ de maneiras de subir $n$ degraus satisfaz

$$E(0) = 1, \qquad E(n) = E(n-1) + E(n-2) + E(n-3) \quad (n \ge 1),$$

com $E(k) = 0$ para $k < 0$. (Pense no **último passo**: ele tem 1, 2 ou 3 degraus. É o mesmo
raciocínio da "última decisão" dos slides.) Complete `escadaMemo()` com memoization. Com a
tabela, `escadaMemo(n)` deve fazer exatamente $3n + 1$ chamadas.

In [ ]:
%%writefile src/escada.c
/* Exercicio 2: DE QUANTAS MANEIRAS SUBIR UMA ESCADA?
   Subindo 1, 2 ou 3 degraus por passo, o numero de maneiras de subir n degraus e
      E(0) = 1,   E(n) = E(n-1) + E(n-2) + E(n-3)   (termos com indice negativo valem 0).
   TODO: complete escadaMemo() com memoization, no mesmo estilo de fibMemo().
   Cada chamada deve incrementar 'chamadas'. */
#include <stdio.h>
#include <stdlib.h>

#define MAX 80

static long long memo[MAX];    /* memo[k] == -1: E(k) ainda nao calculado */
static long long chamadas = 0;

long long escadaMemo( int n) {
   ++chamadas;
   if (n < 0) return 0;
   /* TODO: consulte a tabela, calcule pela recorrencia e anote a resposta */
   return -1;
}

int main( int argc, char *argv[]) {
   int n = argc > 1 ? atoi( argv[1]) : 10;
   for (int k = 0; k < MAX; ++k) memo[k] = -1;
   for (int k = 0; k <= n; ++k) printf( "%lld ", escadaMemo( k));
   printf( "\n");
   for (int k = 0; k < MAX; ++k) memo[k] = -1;
   chamadas = 0;
   long long r = escadaMemo( n);
   printf( "n=%d maneiras=%lld chamadas=%lld\n", n, r, chamadas);
   return 0;
}

In [ ]:
# Teste automático do Exercício 2
saida = rodar('escada.c', args=(50,))
primeiros = [int(x) for x in saida.splitlines()[0].split()]
assert primeiros[:11] == [1, 1, 2, 4, 7, 13, 24, 44, 81, 149, 274], 'Valores errados para n = 0..10'
assert campo(saida, 'maneiras') == 10562230626642, 'E(50) errado'
assert campo(saida, 'chamadas') == 3 * 50 + 1, 'Sem memoization o número de chamadas explode'
print('Exercício 2 ok!')

## 3. Subestrutura ótima: quando a recorrência está correta

Um problema tem **subestrutura ótima** se toda solução ótima contém soluções ótimas dos
subproblemas que usa. Caminho mínimo tem: se um subcaminho $u \leadsto x$ de um caminho
mínimo não fosse mínimo, trocá-lo por um mais curto encurtaria o caminho todo (recorta e cola).

Caminho **simples mais longo** não tem. No grafo dos slides (Cormen et al., Figura 15.6), com
arestas nos dois sentidos entre $q$–$r$, $q$–$s$, $r$–$t$ e $s$–$t$, o mais longo de $q$ a $t$ é
$q \to r \to t$, mas o mais longo de $q$ a $r$ é $q \to s \to t \to r$.

Aqui conferimos as duas afirmações por **força bruta**, enumerando todos os caminhos simples.
Só funciona em grafos pequenos, mas é ótimo para testar intuições.

**Fonte:** Cormen et al. (2009), §15.3 e Figura 15.6; Lema 24.1 (subcaminhos de caminhos mínimos).

In [ ]:
# Exemplo: enumerando caminhos simples e conferindo a subestrutura ótima do caminho MÍNIMO.
from itertools import permutations

G = {'q': ['r', 's'], 'r': ['q', 't'], 's': ['q', 't'], 't': ['r', 's']}   # grafo dos slides

def caminhos_simples(g, u, v, visitados=None):
    """Gera todos os caminhos simples de u a v (listas de vértices)."""
    visitados = (visitados or []) + [u]
    if u == v:
        yield visitados
        return
    for w in g[u]:
        if w not in visitados:
            yield from caminhos_simples(g, w, v, visitados)

def caminho_minimo(g, u, v):
    return min(caminhos_simples(g, u, v), key=len)

# Em um grafo maior (uma grade 3x3 com uma diagonal), todo subcaminho de um mínimo é mínimo.
H = {}
for i in range(3):
    for j in range(3):
        viz = [(i + di, j + dj) for di, dj in [(0, 1), (1, 0), (0, -1), (-1, 0)]
               if 0 <= i + di < 3 and 0 <= j + dj < 3]
        H[(i, j)] = viz
H[(0, 0)].append((1, 1)); H[(1, 1)].append((0, 0))

for u, v in permutations(H, 2):
    c = caminho_minimo(H, u, v)
    for a in range(len(c)):
        for b in range(a + 1, len(c)):
            assert len(c[a:b + 1]) == len(caminho_minimo(H, c[a], c[b])), (u, v, a, b)
print('Caminho mínimo: todos os subcaminhos conferidos são mínimos.')
print('Mínimo de q a t no grafo dos slides:', ' -> '.join(caminho_minimo(G, 'q', 't')))

In [ ]:
# Exercício 3: caminho simples mais longo não tem subestrutura ótima
# TODO: implemente caminho_mais_longo(g, u, v), que devolve UM caminho simples de u a v com o
# maior número de arestas (lista de vértices). Use caminhos_simples() da célula anterior.

def caminho_mais_longo(g, u, v):
    # TODO: implemente aqui
    pass

# Teste automático
qt = caminho_mais_longo(G, 'q', 't')
assert qt is not None, 'Implemente caminho_mais_longo()'
qr = caminho_mais_longo(G, 'q', 'r')
rt = caminho_mais_longo(G, 'r', 't')
assert len(qt) - 1 == 2, 'O mais longo de q a t tem 2 arestas'
assert len(qr) - 1 == 3 and len(rt) - 1 == 3, 'Os mais longos de q a r e de r a t têm 3 arestas'
juntos = qr + rt[1:]
assert len(set(juntos)) < len(juntos), 'A concatenação dos dois ótimos deveria repetir vértice'
print('Exercício 3 ok!')
print('q->r mais longo:', ' -> '.join(qr), '| r->t mais longo:', ' -> '.join(rt))
print('Juntando os dois:', ' -> '.join(juntos), '(não é simples!)')

## 4. Seleção de intervalos ponderados

Intervalos $1, \dots, n$ ordenados por término, com início $s_j$, término $f_j$ e valor $v_j$.
$p(j)$ é o maior $i < j$ com $f_i \le s_j$ (0 se não houver). A recorrência é

$$\mathrm{OPT}(0) = 0, \qquad \mathrm{OPT}(j) = \max\bigl(v_j + \mathrm{OPT}(p(j)),\ \mathrm{OPT}(j-1)\bigr).$$

O programa abaixo implementa os quatro passos: ordena por término, calcula $p$ por busca
binária, preenche $M$ de três jeitos (recursão direta, memoizada e bottom-up) e reconstrói
a solução. Ele também tem um modo `./intervalos n semente` que gera a instância "ruim" em que
$p(j) = j - 2$ (o intervalo $j$ é $[j, j+2)$): ali a recursão direta tem a árvore de Fibonacci.

**Fonte:** Kleinberg e Tardos (2006), §6.1 (recorrência, memoization, `Find-Solution`) e §6.2 (bottom-up).

In [ ]:
%%writefile src/intervalos.c
/* SELECAO DE INTERVALOS PONDERADOS por programacao dinamica.
   Uso: ./intervalos < entrada        (n, depois n linhas "s f v"; imprime p, M e a solucao)
        ./intervalos n semente        (instancia "Fibonacci": [j, j+2), logo p(j) = j-2;
                                       compara as chamadas da recursao direta e da memoizada)
   Kleinberg e Tardos (2006), secs. 6.1 e 6.2. */
#include <stdio.h>
#include <stdlib.h>

typedef struct { int s, f, v, id; } Intervalo;

static int n;
static long long *v, *M;       /* v[1..n], M[0..n]; M[j] == -1: ainda nao calculado */
static int *s, *f, *p;          /* s[1..n], f[1..n], p[1..n] */
static long long chamIngenuo = 0, chamMemo = 0;

static int porTermino( const void *a, const void *b) {
   const Intervalo *x = a, *y = b;
   return (x->f > y->f) - (x->f < y->f);
}

/* p[j] = maior i < j com f[i] <= s[j] (0 se nao houver), por busca binaria em f[1..j-1] */
static void calculaP( void) {
   for (int j = 1; j <= n; ++j) {
      int ini = 0, fim = j - 1;          /* invariante: f[ini] <= s[j] (f[0] = -inf) */
      while (ini < fim) {
         int meio = (ini + fim + 1) / 2;
         if (f[meio] <= s[j]) ini = meio;
         else fim = meio - 1;
      }
      p[j] = ini;
   }
}

static long long max( long long a, long long b) { return a >= b ? a : b; }

/* recursao direta: OPT(j) = max( v[j] + OPT(p[j]), OPT(j-1)) */
long long optIngenuo( int j) {
   ++chamIngenuo;
   if (j == 0) return 0;
   return max( v[j] + optIngenuo( p[j]), optIngenuo( j-1));
}

/* a mesma recursao, memoizada */
long long optMemo( int j) {
   ++chamMemo;
   if (j == 0) return 0;
   if (M[j] >= 0) return M[j];
   long long com = v[j] + optMemo( p[j]);
   long long sem = optMemo( j-1);
   return M[j] = max( com, sem);
}

/* bottom-up: M[j] depende so de M[p[j]] e M[j-1], ja preenchidos */
long long optBU( void) {
   M[0] = 0;
   for (int j = 1; j <= n; ++j)
      M[j] = max( v[j] + M[p[j]], M[j-1]);
   return M[n];
}

/* refaz as decisoes a partir da tabela M (preenchida) */
void solucao( int j) {
   while (j > 0) {
      if (v[j] + M[p[j]] >= M[j-1]) {
         printf( " %d", j);
         j = p[j];
      }
      else j = j - 1;
   }
}

static void aloca( void) {
   v = malloc( (n+1) * sizeof (long long));  M = malloc( (n+1) * sizeof (long long));
   s = malloc( (n+1) * sizeof (int));  f = malloc( (n+1) * sizeof (int));
   p = malloc( (n+1) * sizeof (int));
   f[0] = -1;  /* sentinela: nunca usado na comparacao (ini = 0 e o caso "nenhum") */
}

int main( int argc, char *argv[]) {
   if (argc >= 3) {                        /* instancia em que p(j) = j-2 */
      n = atoi( argv[1]);
      srand( atoi( argv[2]));
      aloca();
      for (int j = 1; j <= n; ++j) { s[j] = j; f[j] = j + 2; v[j] = 1 + rand() % 9; }
      calculaP();
      for (int j = 0; j <= n; ++j) M[j] = -1;
      long long a = optMemo( n);
      long long b = n <= 40 ? optIngenuo( n) : -1;   /* acima disso demora demais */
      long long c = optBU();
      printf( "n=%d opt=%lld ingenuo=%lld bottomup=%lld chamadas_memo=%lld chamadas_ingenuo=%lld\n",
              n, a, b, c, chamMemo, b >= 0 ? chamIngenuo : -1);
      return 0;
   }
   if (scanf( "%d", &n) != 1) return 1;
   Intervalo *I = malloc( n * sizeof (Intervalo));
   for (int i = 0; i < n; ++i) {
      if (scanf( "%d %d %d", &I[i].s, &I[i].f, &I[i].v) != 3) return 1;
      I[i].id = i + 1;
   }
   qsort( I, n, sizeof (Intervalo), porTermino);    /* passo 0: ordenar por termino */
   aloca();
   for (int j = 1; j <= n; ++j) { s[j] = I[j-1].s; f[j] = I[j-1].f; v[j] = I[j-1].v; }
   calculaP();
   for (int j = 0; j <= n; ++j) M[j] = -1;
   long long a = optMemo( n);
   long long b = optIngenuo( n);
   long long c = optBU();
   printf( "p:");  for (int j = 1; j <= n; ++j) printf( " %d", p[j]);  printf( "\n");
   printf( "M:");  for (int j = 0; j <= n; ++j) printf( " %lld", M[j]); printf( "\n");
   printf( "solucao:");  solucao( n);  printf( "\n");
   printf( "opt=%lld ingenuo=%lld bottomup=%lld chamadas_memo=%lld chamadas_ingenuo=%lld\n",
           a, b, c, chamMemo, chamIngenuo);
   return 0;
}

In [ ]:
# Exemplo: a instância dos slides.
saida = rodar('intervalos.c', entrada=INSTANCIA)
assert linha(saida, 'p') == [0, 0, 1, 0, 3, 3]
assert linha(saida, 'M') == [0, 1, 6, 6, 7, 8, 8]
assert sorted(linha(saida, 'solucao')) == [2, 5] and campo(saida, 'opt') == 8
print('Instância dos slides conferida: solução {2, 5}, valor 8.')

In [ ]:
# Visualização: chamadas da recursão direta x memoizada na instância com p(j) = j - 2.
cham_ing, cham_memo = {}, {}
for n in range(4, 33, 2):
    s = rodar('intervalos.c', args=(n, 1), mostrar=False)
    assert campo(s, 'opt') == campo(s, 'ingenuo') == campo(s, 'bottomup')
    cham_ing[n], cham_memo[n] = campo(s, 'chamadas_ingenuo'), campo(s, 'chamadas_memo')

ns = sorted(cham_ing)
plt.figure(figsize=(6, 4))
plt.semilogy(ns, [cham_ing[n] for n in ns], 'o-', label='recursão direta')
plt.semilogy(ns, [cham_memo[n] for n in ns], 's-', label='memoizada: 2n + 1')
plt.xlabel('n')
plt.ylabel('chamadas de OPT (escala log)')
plt.title('Intervalos ponderados com p(j) = j - 2')
plt.legend()
plt.grid(True, which='both', alpha=0.3)
plt.show()

### Exercício 4: o vetor $p$ por busca binária

Calcular $p(j)$ comparando $j$ com todos os anteriores custa $O(n^2)$. Como os términos estão
ordenados, o predicado "$f_i \le s_j$" é verdadeiro para um prefixo $1, \dots, k$ e falso
depois: $p(j)$ é o fim desse prefixo, e uma **busca binária** o encontra em $O(\log n)$.

Complete `calculaP()` abaixo. O `main` compara a sua resposta com a força bruta em 500
instâncias aleatórias, com términos repetidos e intervalos que começam antes de zero.
Cuidado com o caso $p(j) = 0$ e com términos iguais.

In [ ]:
%%writefile src/calcula_p.c
/* Exercicio 4: O VETOR p POR BUSCA BINARIA.
   Os intervalos 1..n estao ordenados por termino: f[1] <= ... <= f[n].
   p[j] = maior i < j com f[i] <= s[j]; p[j] = 0 se nenhum i < j for compativel com j.
   TODO: complete calculaP() em O(n log n): para cada j, uma busca binaria em f[1..j-1].
   O main confere a sua resposta com a forca bruta O(n^2) em instancias aleatorias. */
#include <stdio.h>
#include <stdlib.h>

void calculaP( int n, const int s[], const int f[], int p[]) {
   for (int j = 1; j <= n; ++j) {
      /* TODO: busca binaria; o resultado vai em p[j] */
      p[j] = 0;
   }
}

static void forcaBruta( int n, const int s[], const int f[], int p[]) {
   for (int j = 1; j <= n; ++j) {
      p[j] = 0;
      for (int i = 1; i < j; ++i) if (f[i] <= s[j]) p[j] = i;
   }
}

static int cmp( const void *a, const void *b) { return *(const int *) a - *(const int *) b; }

int main( void) {
   int ok = 1;
   for (int teste = 0; teste < 500 && ok; ++teste) {
      int n = 1 + rand() % 40;
      int s[41], f[41], p[41], q[41], dur[41];
      for (int j = 1; j <= n; ++j) f[j] = rand() % 50;      /* terminos com repeticoes */
      qsort( f + 1, n, sizeof (int), cmp);
      for (int j = 1; j <= n; ++j) { dur[j] = 1 + rand() % 12; s[j] = f[j] - dur[j]; }
      calculaP( n, s, f, p);
      forcaBruta( n, s, f, q);
      for (int j = 1; j <= n; ++j)
         if (p[j] != q[j]) {
            printf( "ERRO: n=%d j=%d s=%d f=%d: esperado p=%d, obtido %d\n", n, j, s[j], f[j], q[j], p[j]);
            ok = 0;
            break;
         }
   }
   printf( "ok=%d\n", ok);
   return 0;
}

In [ ]:
# Teste automático do Exercício 4
saida = rodar('calcula_p.c')
assert campo(saida, 'ok') == 1, 'Verifique sua implementação (veja o ERRO impresso acima)'
print('Exercício 4 ok!')

In [ ]:
# Exercício 5: a força bruta como referência
# TODO: implemente forca_bruta(intervalos), que recebe uma lista de tuplas (s, f, v) e devolve
# o maior valor total de um subconjunto de intervalos dois a dois disjuntos, testando TODOS os
# 2^n subconjuntos. Dois intervalos [s1, f1) e [s2, f2) são disjuntos se f1 <= s2 ou f2 <= s1.
# Depois, o teste compara a sua força bruta com o programa em C em instâncias aleatórias.
import random
from itertools import combinations

def forca_bruta(intervalos):
    # TODO: implemente aqui
    pass

# Teste automático
slides = [(0, 4, 1), (1, 6, 6), (5, 7, 4), (2, 9, 7), (8, 10, 2), (7, 11, 1)]
assert forca_bruta(slides) == 8, 'A instância dos slides tem ótimo 8'
random.seed(2026)
for _ in range(60):
    n = random.randint(1, 12)
    inst = []
    for _ in range(n):
        s = random.randint(0, 20)
        inst.append((s, s + random.randint(1, 8), random.randint(1, 10)))
    entrada = f'{n}\n' + ''.join(f'{s} {f} {v}\n' for s, f, v in inst)
    pd = campo(rodar('intervalos.c', entrada=entrada, mostrar=False), 'opt')
    assert forca_bruta(inst) == pd, f'Divergência na instância {inst}'
print('Exercício 5 ok! A PD concorda com a força bruta em 60 instâncias aleatórias.')

## Desafio Final: a fileira de moedas

Há uma fileira de $n$ moedas com valores $c_1, \dots, c_n$ (inteiros positivos, não
necessariamente distintos). Pegue um subconjunto de valor máximo sem pegar **duas moedas
vizinhas**. Exemplo (Levitin, 2012, §8.1): para $5, 1, 2, 10, 6, 2$ o máximo é $17$, com as
moedas $5$, $10$ e $2$ (posições 1, 4 e 6).

1. **Estrutura e recorrência.** Olhe para a última moeda e escreva a recorrência de $F(n)$, o
   valor máximo usando as moedas $1, \dots, n$. Prove a subestrutura ótima pelo argumento
   recorta e cola, como nos slides.
2. **Redução.** Mostre que a fileira de moedas é um caso particular de intervalos ponderados:
   que intervalo $[s_j, f_j)$ você associa à moeda $j$? Quanto vale $p(j)$? Confira rodando
   `src/intervalos.c` na instância do exemplo.
3. **Implementação.** Escreva `src/moedas.c` com a versão bottom-up em $O(n)$ tempo e a
   reconstrução das posições escolhidas. Teste com o exemplo e com $n = 10^6$ moedas aleatórias.
4. **Espaço.** Reescreva o cálculo do valor com espaço $O(1)$, como no `fibBU` com duas
   variáveis. Por que a reconstrução fica mais difícil nessa versão?

In [ ]:
# Desafio: a fileira de moedas
# TODO: resolva o desafio abaixo
# Sugestão: comece pelo item 2, montando a entrada de src/intervalos.c para 5 1 2 10 6 2.

# Sua solução aqui

## Referências

Veja o arquivo `../referencias.bib` para a lista completa. As principais para este tutorial:

- CORMEN, T. H. et al. *Introduction to Algorithms*. 3. ed. MIT Press, 2009. Cap. 15 (§15.1 e §15.3) e Lema 24.1. (Em português: *Algoritmos: Teoria e Prática*, Elsevier, 2012.)
- KLEINBERG, J.; TARDOS, É. *Algorithm Design*. Pearson/Addison-Wesley, 2006. §§6.1–6.2.
- LEVITIN, A. *Introduction to the Design and Analysis of Algorithms*. 3. ed. Pearson, 2012. Cap. 8 e §8.1.
- DASGUPTA, S.; PAPADIMITRIOU, C. H.; VAZIRANI, U. *Algorithms*. McGraw-Hill, 2008. §0.2 e §6.1.
- FEOFILOFF, P. *Programação dinâmica*. Análise de Algoritmos, IME-USP. https://www.ime.usp.br/~pf/analise_de_algoritmos/aulas/dynamic-programming.html
- MICHIE, D. "Memo" functions and machine learning. *Nature*, v. 218, p. 19–22, 1968. DOI 10.1038/218019a0.